In [1]:
# input
pred_file = "../../data/pred_ge_3_clique_3.tsv"
seq_file = "../../tmp/pred_ge_3_clique_3.fasta"

In [2]:
from tqdm import tqdm
from Bio import SeqIO
import pandas as pd

df = pd.read_table(pred_file)
seq2posi = dict(zip(df['seq_id'], df['posi']))

In [3]:
record = []
for r in tqdm(SeqIO.parse(seq_file, "fasta"), total=len(df)):
    seq_id = r.id
    seq = str(r.seq)
    positions = seq2posi[seq_id].split(",")
    residues = [seq[int(i)] for i in positions]
    record.append({
        "seq_id": seq_id,
        "resi": ",".join(residues)
    })
df = pd.merge(df, pd.DataFrame(record), on="seq_id")
del seq2posi

  0%|          | 0/38361041 [00:00<?, ?it/s]

100%|██████████| 38361041/38361041 [04:19<00:00, 147748.35it/s]


In [13]:
import numpy as np

probas = list(map(float, ",".join(df['pred']).split(",")))
resis = ",".join(df['resi']).split(",")

probas = np.array(probas)
resis = np.array(resis)

In [15]:
from collections import Counter
records = []
for i in range(2, 10):
    p = i / 10

    counter = Counter(resis[probas >= p])
    counter_dict = dict(counter)
    total = sum(counter_dict.values())
    proportions = {f"{residue}_ratio": count / total for residue, count in counter_dict.items()}
    counter_dict.update(proportions)
    counter_dict['proba'] = p
    records.append(counter_dict)

In [16]:
pd.DataFrame(records)

,E,H,C,D,E_ratio,H_ratio,C_ratio,D_ratio,proba
0,20596689,52642585,88496441,45605184,0.099337,0.253894,0.426816,0.219953,0.2
1,17932795,49239145,86105473,41032613,0.092290,0.253405,0.443134,0.211171,0.3
2,16253669,46880283,84156725,37748321,0.087839,0.253354,0.454805,0.204002,0.4
3,14855062,44766793,82329874,35146466,0.083880,0.252779,0.464883,0.198458,0.5
4,13410935,42503774,80155531,32464636,0.079574,0.252196,0.475602,0.192629,0.6
5,11574194,39398664,76900955,29090838,0.073738,0.251003,0.489925,0.185334,0.7
6,8709340,34051049,71091093,23419441,0.063446,0.248057,0.517889,0.170607,0.8
7,5272314,24500363,57408170,15439415,0.051377,0.238748,0.559423,0.150452,0.9
